# diag colab_11 — where did v2's fit actually land? A LoRA ablation ladder

`colab_11` v2 added `dense` to the LoRA target list and reached a slightly better loss than the
attention-only v1 pilot, while detector #1's embedding drift did **not** move (0.0057 → 0.0051).
Loss-fit and cosine-drift decoupled. This notebook asks where v2's extra fit physically went, and
whether detector #1 was positioned to see it.

**The geometry that motivates this.** `Geneformer-V2-104M` is a 12-layer BERT (hidden 768,
intermediate 3072). Embeddings are extracted with `emb_layer=-1`, which in Geneformer's convention
means the **2nd-to-last** layer — the output of layer index 10. Everything after that point is
invisible to detector #1: the whole of encoder layer 11, plus the MLM head. `dense` matched **37**
modules — 36 encoder projections and one head projection, `cls.predictions.transform.dense` — so
v2's added capacity is overwhelmingly encoder-side, and only a thin slice of it sits in the blind
spot:

| region | LoRA params | share of v2's trainable |
|---|---|---|
| MLM head (`cls.predictions.transform.dense`) | 12,288 | 0.9% |
| encoder layer 11 (qkv + all dense) | 110,592 | 8.3% |
| **both — detector #1's blind spot** | **122,880** | **9.2%** |
| v2 total trainable | 1,339,392 | 100% |

So detector #1 sees roughly 91% of the adapted capacity. The head-absorption hypothesis — that v2's
loss gain landed in the MLM head, downstream of the measurement — requires 0.9% of the parameters to
carry essentially all of the gain. Possible, since the head sits at the output where per-parameter
leverage on the logits is greatest, but it is a strong claim. This notebook tests it directly instead
of arguing about it.

**Method.** No retraining. Both LoRA adapters are on Drive; each variant is re-merged from the frozen
base checkpoint, with selected LoRA deltas zeroed, and evaluated on one **frozen masked validation
set** — the same masked tokens for every variant, so the only thing varying is model weights.

**The ladder.**

| rung | variant | question |
|---|---|---|
| 0 | base / v1 / v2 intact | is there a gain to attribute at all? |
| A | v2 − head | head absorption proper |
| B | v2 − layer 11 | last-encoder-layer absorption |
| C | v2 − both | how much of v2's fit lives in detector #1's blind spot |

**Two limits, stated up front.**

1. *Ablation is not decomposition.* The encoder trained **with** the head delta present. Zeroing it
   puts the model slightly off-distribution, so loss can land above v1's rather than reverting to it.
   Each rung yields an **upper bound** on its region's contribution, not an additive share, and the
   rungs need not sum to C.
2. *This cannot tell us the gap is real.* colab_11's validation curve has v2 lower at all 8
   checkpoints, widening from 0.0031 to 0.0049. That is weaker evidence than it looks: the 8
   checkpoints are 8 samples of the **same two trajectories**, near-perfectly autocorrelated, so
   "v2 wins 8/8" is close to one observation, not eight. v1 and v2 are N=1 each — they share
   `SEED=0`, but adding `dense` changes the trainable-parameter set, so LoRA init and dropout draws
   diverge and the runs are different trajectories. Separating "`dense` helps" from "this seed's
   trajectory" needs retraining, which is the cost this arc exists to avoid. §7b bounds only
   **measurement** noise — but that is not nothing here: v1 and v2 consumed different training RNG,
   so their eval masks likely differed too, and if the mask-draw spread turns out to be of order
   0.005 then the reference curve itself was never readable and there is no decoupling to explain.

Everything is measured on **val**, never test. Val is currently used only for CPT loss, so it is free
for this; choosing a representation or an interpretation on test and then running the contract's evals
on test would be selection on the evaluation surface.

## 1 — Setup

### 1a — Mount Drive, clone the repo, install the Geneformer environment

**A later cell will crash from this — restart the runtime when this cell finishes.** The installs
below upgrade numpy on disk, but Colab's kernel already imported numpy before the cell ran, so the
in-memory module and its freshly-upgraded compiled extensions disagree. §3a's `import anndata` then
fails on a numpy internal (`AttributeError: _blas_supports_fpe` was the live symptom in colab_11).
This is the identical install path colab_11 documents, so it is the identical crash. **Runtime >
Restart session** after this cell, then run from §1b onward — the installs persist across a restart,
so nothing above needs re-running.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
assert os.path.exists(DRIVE_ROOT), f"Drive root not found: {DRIVE_ROOT}"

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

# Lean Geneformer-only stack: rides Colab's native numpy-2 base (see NUMPY POLICY in
# requirements_geneformer.txt); scGPT is NOT installed here.
!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .

# Colab preinstalls torchao 0.10.0, unused here (no quantization, plain LoRA). peft's
# get_peft_model dispatches every target Linear through dispatch_torchao(), which calls
# is_torchao_available() -- if torchao is merely *importable*, that function hard-raises
# ImportError below its 0.16.0 floor rather than returning False. Uninstalling (not upgrading) is
# the minimal fix. Same treatment as colab_11, which this notebook must stay binary-comparable to.
!pip uninstall -y torchao

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True).stdout.strip()
print("Geneformer commit:", GENEFORMER_COMMIT)

### 1b — Run configuration

One `SMOKE` toggle drives everything. `SMOKE = True` subsamples the validation split to a few hundred
cells and collapses the noise-floor draws to two, so the whole notebook can be walked end-to-end in a
few minutes to prove the plumbing before an A100 hour is spent on it. Every output path is *derived*
from the toggle and carries a `_SMOKE` suffix, so a smoke run can waste time but can never overwrite a
real result. The split check in §3b runs **before** any subsampling, so it is never weakened by it.

In [ ]:
from datetime import date
TODAY = date.today().isoformat()

SMOKE = False            # live toggle -- committed False; flip in the Colab tab for a dry run

SEED           = 0       # matches colab_11's training seed
MLM_PROB       = 0.15    # MUST match CPT -- Geneformer's pretraining default
EVAL_BATCH     = 8       # NOT a free knob -- do not tune for speed. Two reasons it is load-
                         # bearing. (1) colab_11 never set per_device_eval_batch_size, so its eval
                         # ran at HF's default 8; matching it is what lets §7a's trainer-style
                         # estimator be compared to colab_11's curve at all. (2) The collator builds
                         # probability_matrix = torch.full(labels.shape, p) over the batch's PADDED
                         # width and draws a bernoulli per element, so batch size changes how many
                         # RNG draws are consumed and how they align -- i.e. it changes the mask
                         # itself, and therefore every loss. Constant across variants, so the ladder
                         # is unaffected either way, but the numbers are not batch-size invariant.
N_MASK_SEEDS   = 4       # §7b: independent mask draws used to bound measurement noise
SMOKE_N_CELLS  = 512

# Reference numbers from colab_11 -- the run this notebook interrogates.
REF_TRAIN_LOSS = {"v1": 2.0971, "v2": 2.0921}

# colab_11's step-matched validation curve. v2 is lower at all 8 checkpoints and the gap widens
# monotonically (0.0031 -> 0.0049). Two cautions on reading that as a real effect: (a) these are 8
# samples of the SAME two trajectories, not 8 independent draws -- consecutive checkpoints are
# near-perfectly autocorrelated, so "v2 wins 8/8" carries almost no evidential weight; (b) v1 and
# v2 consumed different amounts of RNG during training (different param counts -> different dropout
# draws), so their eval masks almost certainly differed, and part of this gap may be mask noise.
# §7b measures the mask-draw spread and therefore calibrates whether this curve was ever readable.
REF_VAL_LOSS = {
    250:  {"v1": 2.093421, "v2": 2.090349},
    500:  {"v1": 2.091376, "v2": 2.088032},
    750:  {"v1": 2.089601, "v2": 2.085899},
    1000: {"v1": 2.089767, "v2": 2.086088},
    1250: {"v1": 2.088168, "v2": 2.084011},
    1500: {"v1": 2.087294, "v2": 2.082720},
    1750: {"v1": 2.086892, "v2": 2.082163},
    2000: {"v1": 2.086208, "v2": 2.081286},
}
REF_VAL_FINAL = REF_VAL_LOSS[2000]   # §7a reproduction target for the trainer-style estimator.
                                     # Mask draw is the ONLY residual difference -- see §7a.

RUN_TAG = "head_ablation" + ("_SMOKE" if SMOKE else "")

GF_DIR       = os.path.join(DRIVE_ROOT, "geneformer")
V1_ADAPTER   = os.path.join(GF_DIR, "cpt_aggregated_seed0_adapter")       # v1: query,key,value
V2_ADAPTER   = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")    # v2: + dense
MODEL_DIR    = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")

for p in (V1_ADAPTER, V2_ADAPTER, MODEL_DIR):
    assert os.path.exists(p), f"missing required input: {p}"

RESULTS_PATH = os.path.join(GF_DIR, f"cpt_{RUN_TAG}_results.json")   # Drive: survives the VM

# The freeze/env are the run's only unrecoverable artifacts, so they go where every other notebook in
# this project puts them -- the repo's version record -- not just on Drive. colab_00..colab_11 all
# write outputs/software_versions/<id>_<date>_pip_freeze.txt; a diag that writes elsewhere silently
# leaves a hole in the Methods record.
VERSIONS_DIR  = os.path.join(REPO_PATH, "outputs", "software_versions")
NOTEBOOK_ID   = "diag_colab_11_head_ablation" + ("_SMOKE" if SMOKE else "")
FREEZE_PATH   = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
AUDIT_KEY    = f"geneformer_cpt_{RUN_TAG}"

if SMOKE:
    # 2, not 1: §7b is the arc's real gate, so the smoke run must actually execute its loop,
    # its spread arithmetic and NOISE_SUMMARY's populated shape. At 1 the whole block is dead
    # code and the run proves nothing about the thing it exists to prove.
    N_MASK_SEEDS = 2
    print("*** SMOKE RUN *** val subsampled to", SMOKE_N_CELLS,
          "cells;", N_MASK_SEEDS, "mask draws; writes ->", os.path.basename(RESULTS_PATH))
    print("*** results are plumbing evidence ONLY -- not a scientific readout ***")
else:
    print("LIVE RUN -> full val split,", N_MASK_SEEDS, "mask draws")

print("run tag:", RUN_TAG, "| date:", TODAY)

## 2 — Environment capture

### 2a — pip freeze + env JSON

The exact stack this measurement ran on. `pip_freeze.txt` is the one genuinely unrecoverable
artifact of a Colab session, so it is written to Drive before any compute.

In [ ]:
import json, platform, subprocess, sys, torch
import peft, transformers, datasets

os.makedirs(GF_DIR, exist_ok=True)
os.makedirs(VERSIONS_DIR, exist_ok=True)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True, check=True).stdout
with open(FREEZE_PATH, "w") as f:
    f.write(freeze)

# Binary comparability with colab_11 is this notebook's whole premise: it re-merges colab_11's
# adapters and reports differences of ca. 0.005 nats. A change in peft's merge semantics or target
# matching, or in transformers' BERT internals, between colab_11's run and this one would move every
# number in the ladder with no crash and no visible sign. colab_11's exact resolved versions are not
# a matter of memory -- they are recorded in outputs/software_versions/colab_11_2026-07-15_pip_freeze.txt
# and are asserted here.
REF_VERSIONS = {"peft": "0.19.1", "transformers": "4.57.0", "datasets": "5.0.0"}
_got = {"peft": peft.__version__, "transformers": transformers.__version__,
        "datasets": datasets.__version__}
_skew = {k: (v, _got[k]) for k, v in REF_VERSIONS.items() if _got[k] != v}
assert not _skew, (
    f"version skew vs colab_11 (expected, got): {_skew}. colab_11's adapters were written under "
    f"{REF_VERSIONS}; re-merging them under different versions can silently change every loss in the "
    f"ladder. Pin these in requirements_geneformer.txt rather than proceeding.")
print("version guard vs colab_11's recorded freeze:", _got, "-- OK")

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
ENV = {
    "date": TODAY, "run_tag": RUN_TAG, "smoke": SMOKE,
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": gpu,
    "peft": peft.__version__, "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "geneformer_commit": GENEFORMER_COMMIT,
}
with open(ENV_JSON_PATH, "w") as f:
    json.dump(ENV, f, indent=2)

print("pip freeze ->", FREEZE_PATH, f"({len(freeze.splitlines())} pkgs)")
print("env        ->", ENV_JSON_PATH)
print("GPU:", gpu, "| torch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU -- this notebook needs one (Runtime > Change runtime type)"

DEVICE = "cuda"

## 3 — Substrate + frozen split

### 3a — Load the glia substrate, guard raw counts

The same cell-for-cell substrate every FM notebook loads: 142,588 glia (54,805 microglia / 87,783
astrocytes), concatenated in the same order with the same `cell_index` key, so tokenization here is
bit-identical to colab_11's.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()

# concat order and index_unique MUST match colab_11 -- cell_index is assigned positionally and is
# the alignment key shared with the zero-shot baseline and every CPT embedding.
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("\ncombined glia:", glia.shape)

assert glia.n_obs == 142588, f"substrate is {glia.n_obs} cells, expected 142588 -- upstream changed"

# raw-counts guard -- Geneformer rank-encodes RAW counts; log-normalized input is silently wrong.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))

### 3b — Apply the frozen donor split and verify it against the reference

The standing split rule: this notebook **loads** `outputs/donor_split.json` and re-derives nothing.
A mismatch against the committed reference is a hard stop, not a licence to mint a new split — a
silently different val set would make every loss here incomparable to colab_11's.

In [ ]:
SPLIT_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_PATH), f"missing committed split {SPLIT_PATH}"
with open(SPLIT_PATH) as f:
    split_meta = json.load(f)

glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_meta["donor_split"])
unmapped = int(glia.obs["split"].isna().sum())
assert unmapped == 0, f"{unmapped} cells whose donor is absent from the frozen split"

REF = {"seed": 32, "margin": 10,
       "donors": {"train": 101, "val": 22, "test": 22},
       "cells":  {"train": 94963, "val": 23824, "test": 23801}}

obs_donors = {k: int(glia.obs.loc[glia.obs["split"] == k, "donor_id"].nunique())
              for k in ("train", "val", "test")}
obs_cells  = {k: int((glia.obs["split"] == k).sum()) for k in ("train", "val", "test")}

print("seed:", split_meta["seed"], "| test worst-case margin:", split_meta["test_worst_case_margin"])
print("donors:", obs_donors)
print("cells: ", obs_cells)

test_studies = (glia.obs.loc[glia.obs["split"] == "test", "study_id"]
                .value_counts(normalize=True).round(3).to_dict())
print("test study balance:", test_studies)

assert split_meta["seed"] == REF["seed"], f"split seed {split_meta['seed']} != reference {REF['seed']}"
assert split_meta["test_worst_case_margin"] == REF["margin"], "split margin != reference"
assert obs_donors == REF["donors"], f"donor counts {obs_donors} != reference {REF['donors']}"
assert obs_cells == REF["cells"], f"cell counts {obs_cells} != reference {REF['cells']}"
print("\nsplit verified against the frozen reference -- OK")

### 3c — Subset to the validation split

Tokenization is provably **cell-local**: the normalization factor comes from Geneformer's shipped
`gene_median_dict`, and the only per-cell quantity is that cell's own `n_counts` (verified against
`tokenizer.py` at the pinned commit — see `docs/ASSUMPTIONS.md`). Nothing depends on which other
cells are present, so tokenizing the val split alone yields tokens bit-identical to tokenizing all
142,588 and subsetting — at a sixth of the cost.

Any `SMOKE` subsample happens **here**, after the §3b verification has already passed on the full
object.

In [ ]:
val = glia[glia.obs["split"] == "val"].copy()
del glia; gc.collect()
print("val split:", val.shape)
print("  lineage:", val.obs["lineage"].value_counts().to_dict())
print("  donors: ", val.obs["donor_id"].nunique())

if SMOKE:
    rng = np.random.default_rng(SEED)
    take = rng.choice(val.n_obs, size=min(SMOKE_N_CELLS, val.n_obs), replace=False)
    val = val[np.sort(take)].copy()
    print(f"*** SMOKE: val subsampled -> {val.n_obs} cells ***")

# Geneformer's tokenizer reads n_counts off obs; compute over the full panel, per cell.
val.obs["n_counts"] = np.asarray(val.X.sum(axis=1)).ravel()
assert (val.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"
print("n_counts median:", float(np.median(val.obs["n_counts"])))

## 4 — Tokenize the validation split

### 4a — Map gene symbols to Ensembl IDs

Same mapping colab_09 and colab_11 used. The APOE hard-fail gate is re-asserted rather than assumed:
if the vocabulary ever shifted under us, this must stop here rather than produce a quietly different
tokenization.

In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
assert "<mask>" in token_dict and "<pad>" in token_dict, "token dictionary missing <mask>/<pad>"

val.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in val.var_names]
in_vocab = val.var["ensembl_id"].map(lambda e: (e in token_dict) if e is not None else False)
n_vocab = int(in_vocab.sum())
print(f"gene panel: {val.n_vars} | in Geneformer vocab: {n_vocab} ({n_vocab/val.n_vars:.1%})")

# Must reproduce colab_09's audited 63.33% -- a drift here means a different vocabulary.
assert abs(n_vocab / val.n_vars - 0.6333) < 0.005, (
    f"vocab coverage {n_vocab/val.n_vars:.4f} departs from colab_09's audited 0.6333")

apoe_e = symbol_to_ensembl.get("APOE")
assert apoe_e is not None and apoe_e in token_dict, "APOE not tokenizable -- pre-registered hard fail"
print("APOE in vocab:", apoe_e, "-- gate passed")

### 4b — Tokenize

Defaults are the ones colab_09 and colab_11 ran on, and are V2-shaped: `special_token=True`,
`model_input_size=4096`, `model_version="V2"` (verified — `docs/ASSUMPTIONS.md`). So `<cls>` is
prepended and `<eos>` appended to every cell, exactly as during CPT.

The `sum_ensembl_ids` monkeypatch is carried over verbatim from colab_11 §4b and is load-bearing:
without it, current pandas raises on the tokenizer's positional-indexing assumption.

In [ ]:
from geneformer import TranscriptomeTokenizer
import geneformer.tokenizer as _gftok

TOK_IN_DIR  = "/content/gf_token_in"
TOK_OUT_DIR = "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True)
os.makedirs(TOK_OUT_DIR, exist_ok=True)
val.write_h5ad(os.path.join(TOK_IN_DIR, "val.h5ad"))

CUSTOM_ATTRS = {
    "cell_index":   "cell_index",
    "split":        "split",
    "lineage":      "lineage",
    "substate":     "substate",
    "apoe_carrier": "apoe_carrier",
    "study_id":     "study_id",
    "donor_id":     "donor_id",
}

# Upstream Geneformer bug (confirmed against the pinned commit's tokenizer.py): tokenize_anndata
# does `adata.var["ensembl_id_collapsed"][coding_miRNA_loc]`, where coding_miRNA_loc is an array of
# integer POSITIONS, but var.index at that point is gene symbols/Ensembl IDs, never integers. Older
# pandas silently fell back to positional indexing; Colab's current pandas raises
# `KeyError: None of [...] are in the [index]`. Reset the post-sum_ensembl_ids var index to a plain
# RangeIndex so positions coincide with labels again -- restores exactly the behaviour this commit's
# code assumes, without touching pandas globally.
_orig_sum_ensembl_ids = _gftok.sum_ensembl_ids

def _sum_ensembl_ids_rangeindex_patch(*args, **kwargs):
    result = _orig_sum_ensembl_ids(*args, **kwargs)
    result.var.index = pd.RangeIndex(result.n_vars)
    return result

_gftok.sum_ensembl_ids = _sum_ensembl_ids_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "val_ablation", file_format="h5ad")

from datasets import load_from_disk
val_ds = load_from_disk(os.path.join(TOK_OUT_DIR, "val_ablation.dataset"))
print("tokenized val:", val_ds)
assert len(val_ds) == val.n_obs, f"tokenizer returned {len(val_ds)} cells, expected {val.n_obs}"

lens = np.array(val_ds["length"])
print(f"token lengths: median {np.median(lens):.0f} | min {lens.min()} | max {lens.max()}")

## 5 — Freeze one masked validation set

### 5a — Materialise fixed masked batches

MLM loss is stochastic in the masking. Comparing variants under independently drawn masks would put
mask noise directly into differences we expect to be on the order of 0.005 nats. So the masking is
done **once**, with Geneformer's own collator, and the resulting tensors are reused verbatim for
every variant: identical batch composition, identical padding, identical masked positions, identical
random-token substitutions. The only thing that varies across the ladder is model weights.

Note a real caveat inherited from upstream: `GeneformerPreCollator` registers only `<mask>` and
`<pad>` as special tokens, so `<cls>` and `<eos>` are ordinary maskable tokens and get masked in
roughly 15% of cells. That is genuine Geneformer CPT behaviour, and reproducing it here is what makes
these losses comparable to colab_11's — but it is a caveat if `<cls>` later becomes the readout.

In [ ]:
import torch
from transformers import DataCollatorForLanguageModeling
from geneformer.pretrainer import GeneformerPreCollator

precollator = GeneformerPreCollator(token_dictionary=token_dict)
collator = DataCollatorForLanguageModeling(tokenizer=precollator, mlm=True, mlm_probability=MLM_PROB)

def build_masked_batches(ds, batch_size, mask_seed):
    """Materialise a fixed set of masked batches. Same seed -> byte-identical batches."""
    torch.manual_seed(mask_seed)
    batches = []
    for i in range(0, len(ds), batch_size):
        rows = ds[i : i + batch_size]["input_ids"]
        batch = collator([{"input_ids": r} for r in rows])
        batches.append({k: v.cpu() for k, v in batch.items()})
    return batches

MASK_SEEDS = [SEED + i for i in range(N_MASK_SEEDS)]
frozen_batches = build_masked_batches(val_ds, EVAL_BATCH, MASK_SEEDS[0])

n_masked = sum(int((b["labels"] != -100).sum()) for b in frozen_batches)
n_tokens = sum(int(b["attention_mask"].sum()) for b in frozen_batches)
print(f"frozen masked set: {len(frozen_batches)} batches | {n_tokens:,} real tokens "
      f"| {n_masked:,} masked ({n_masked/n_tokens:.3%})")
assert abs(n_masked / n_tokens - MLM_PROB) < 0.02, "masking rate departs from MLM_PROB"

# determinism check -- rebuilding with the same seed must reproduce the set exactly, or every
# cross-variant comparison below is resting on an assumption that does not hold.
_check = build_masked_batches(val_ds.select(range(min(len(val_ds), 2 * EVAL_BATCH))), EVAL_BATCH, MASK_SEEDS[0])
for j, b in enumerate(_check):
    assert torch.equal(b["input_ids"], frozen_batches[j]["input_ids"]), \
        "masking is NOT reproducible under a fixed seed -- the ladder would be measuring mask noise"
print("masking determinism verified under fixed seed -- OK")
del _check

## 6 — Build the model variants

### 6a — Ablation machinery: zero a LoRA delta by module name

A LoRA module contributes `scaling * (B @ A)` to its base weight. Zeroing `B` removes that
contribution exactly, so `merge_and_unload()` then folds in a delta of zero and the module reverts to
its frozen pretrained weight. Every variant is loaded fresh from the base checkpoint so no ablation
can leak into the next one.

In [ ]:
from peft import PeftModel
from transformers import BertForMaskedLM

def load_variant(adapter_dir, zero_matching=(), do_merge=True):
    """Re-merge a variant from the frozen base. `zero_matching` = substrings of module names whose
    LoRA delta is zeroed before merging. `do_merge=False` skips merge_and_unload for name/count
    audits that never run a forward pass. Returns (model, {module_name: lora_params_zeroed}, n_lora)."""
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    if adapter_dir is None:
        assert not zero_matching, "cannot ablate the base model -- it carries no adapter"
        return base, {}, 0

    model = PeftModel.from_pretrained(base, adapter_dir)

    lora_modules = [(n, m) for n, m in model.named_modules() if hasattr(m, "lora_B") and len(m.lora_B)]
    n_lora = len(lora_modules)

    # FAIL LOUD: peft's set_peft_model_state_dict does NOT raise when adapter keys fail to match the
    # freshly-constructed base -- it collects them in unexpected_keys and returns. LoRA's B is
    # zero-initialised by construction, so an unloaded adapter leaves every delta at exactly zero and
    # merge_and_unload() folds in nothing: every variant silently becomes the base model, and the
    # ladder reports six identical losses that look plausible. The module-count asserts below cannot
    # catch this -- they count modules built from adapter_config.json's target list, not weights read
    # from adapter_model.safetensors. So check the weights themselves, before anything is zeroed.
    n_live = sum(1 for _, m in lora_modules
                 for k in m.lora_B if float(m.lora_B[k].weight.abs().sum()) > 0)
    n_slots = sum(len(m.lora_B) for _, m in lora_modules)
    assert n_live == n_slots, (
        f"{adapter_dir}: only {n_live}/{n_slots} lora_B tensors are non-zero -- the adapter did not "
        f"fully load onto this base and the merged delta is (partly) empty. Every number below would "
        f"be a base-model number wearing a variant's name.")

    # Zeroing lora_B alone is a COMPLETE ablation only while bias="none". peft merges
    # lora_B[a].bias * scaling into base_layer.bias on a separate path (lora/layer.py merge(),
    # guarded by self.lora_bias[adapter]), which zeroing the weight would not touch -- a future
    # lora_bias=True adapter would ablate partially and silently. Assert the config rather than
    # inherit the assumption.
    with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
        _acfg = json.load(f)
    assert _acfg.get("lora_bias", False) is False and _acfg.get("bias", "none") == "none", (
        f"{adapter_dir}: bias={_acfg.get('bias')} lora_bias={_acfg.get('lora_bias')} -- zeroing "
        f"lora_B would leave a bias delta merged in, so the ablation would be incomplete")

    zeroed = {}
    for name, mod in lora_modules:
        if not any(pat in name for pat in zero_matching):
            continue
        with torch.no_grad():
            for k in mod.lora_B:
                mod.lora_B[k].weight.zero_()
        zeroed[name] = sum(mod.lora_A[k].weight.numel() + mod.lora_B[k].weight.numel()
                           for k in mod.lora_A)

    if zero_matching:
        assert zeroed, f"no LoRA module matched {zero_matching} -- the module tree is not what we think"

    if not do_merge:
        return model, zeroed, n_lora
    merged = model.merge_and_unload()
    return merged, zeroed, n_lora

print("ablation machinery ready")

### 6b — Enumerate variants and verify the ablated parameter counts

This is the cell that proves the ladder is ablating what it claims to. The expected LoRA parameter
counts are **derived from the checkpoint's own config** (`r * (d_in + d_out)` per module) and
asserted against what was actually zeroed. If Geneformer's module tree, `peft`'s leaf-name matching,
or the LoRA rank is not what this notebook assumes, it stops here rather than reporting a confidently
wrong attribution.

In [ ]:
from transformers import BertConfig

cfg = BertConfig.from_pretrained(MODEL_DIR)
H, I, L = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers
R = 8   # colab_11's LORA_R, both runs
print(f"checkpoint: {L} layers | hidden {H} | intermediate {I} | lora r={R}")
assert (L, H, I) == (12, 768, 3072), f"checkpoint geometry changed: {(L, H, I)}"

LAST_LAYER = L - 1                       # emb_layer=-1 extracts the output of layer L-2, so layer
                                         # L-1 is downstream of the measurement point
HEAD_MATCH = "cls.predictions.transform.dense"
LAST_MATCH = f".layer.{LAST_LAYER}."

# expected LoRA params, derived not hardcoded: r*(d_in+d_out) per adapted module
p = lambda d_in, d_out: R * (d_in + d_out)
EXP_HEAD = p(H, H)                                                   # head transform dense
EXP_LAST = 3 * p(H, H) + p(H, H) + p(H, I) + p(I, H)                 # qkv + attn-out + FFN up/down
EXP_BOTH = EXP_HEAD + EXP_LAST

# v2's full trainable count, derived on the same principle rather than hardcoded -- it is the
# denominator for every "% of v2's capacity" claim this notebook makes.
V1_TRAINABLE = 3 * L * p(H, H)                                   # q,k,v across all layers
V2_TRAINABLE = V1_TRAINABLE + L * (p(H, H) + p(H, I) + p(I, H)) + EXP_HEAD
# Cross-check the derivation against what colab_11's own run RECORDED as trainable, not against
# a restatement of the same arithmetic. audit_report.json is the independent record here; if this
# derivation and that record disagree, the geometry assumed by every rung below is wrong.
with open(os.path.join(REPO_PATH, "outputs", "audit_report.json")) as _f:
    _rec = json.load(_f)
REC_TRAINABLE = {"v1": 442368, "v2": 1339392}   # fallback if the run block is absent from the record
for _k in ("geneformer_cpt_aggregated_v2", "geneformer_cpt_aggregated"):
    if _k in _rec and "trainable_params" in _rec[_k]:
        REC_TRAINABLE["v2" if "v2" in _k else "v1"] = _rec[_k]["trainable_params"]
assert V2_TRAINABLE == REC_TRAINABLE["v2"], (
    f"derived v2 trainable {V2_TRAINABLE:,} != colab_11's recorded {REC_TRAINABLE['v2']:,} -- "
    f"the checkpoint geometry or LoRA target set is not what this notebook assumes")
assert V1_TRAINABLE == REC_TRAINABLE["v1"], (
    f"derived v1 trainable {V1_TRAINABLE:,} != colab_11's recorded {REC_TRAINABLE['v1']:,}")
print(f"expected LoRA params -- head {EXP_HEAD:,} | layer {LAST_LAYER} {EXP_LAST:,} | both {EXP_BOTH:,}")
print(f"v2 trainable {V2_TRAINABLE:,} | detector #1 blind spot {EXP_BOTH:,} = {EXP_BOTH/V2_TRAINABLE:.1%} of it")

VARIANTS = [
    ("base",                None,       ()),
    ("v1_intact",           V1_ADAPTER, ()),
    ("v2_intact",           V2_ADAPTER, ()),
    ("v2_minus_head",       V2_ADAPTER, (HEAD_MATCH,)),
    ("v2_minus_last_layer", V2_ADAPTER, (LAST_MATCH,)),
    ("v2_minus_both",       V2_ADAPTER, (HEAD_MATCH, LAST_MATCH)),
]

EXPECT_ZEROED = {"v2_minus_head": EXP_HEAD, "v2_minus_last_layer": EXP_LAST, "v2_minus_both": EXP_BOTH}
EXPECT_N_LORA = {"v1_intact": 3 * L, "v2_intact": 6 * L + 1,
                 "v2_minus_head": 6 * L + 1, "v2_minus_last_layer": 6 * L + 1,
                 "v2_minus_both": 6 * L + 1}

ABLATION_AUDIT = {}
for name, adapter, pats in VARIANTS:
    if adapter is None:
        continue
    # do_merge=False: this is a name/count audit and never runs a forward pass, so the merge
    # is pure cost. It stays a separate pre-flight cell rather than folding into §7a because
    # a module-tree mismatch should fail here, before the first eval pass, not hours in.
    _m, zeroed, n_lora = load_variant(adapter, pats, do_merge=False)
    total = sum(zeroed.values())
    ABLATION_AUDIT[name] = {"n_lora_modules": n_lora, "n_modules_zeroed": len(zeroed),
                            "lora_params_zeroed": total, "modules": sorted(zeroed)}
    print(f"\n{name}: {n_lora} LoRA modules | zeroed {len(zeroed)} module(s) = {total:,} params")
    for mn in sorted(zeroed):
        print(f"    {mn}  ({zeroed[mn]:,})")
    assert n_lora == EXPECT_N_LORA[name], \
        f"{name}: {n_lora} LoRA modules, expected {EXPECT_N_LORA[name]} -- target matching changed"
    if name in EXPECT_ZEROED:
        assert total == EXPECT_ZEROED[name], \
            f"{name}: zeroed {total:,} params, expected {EXPECT_ZEROED[name]:,}"
    del _m; gc.collect(); torch.cuda.empty_cache()

print("\nablation targets verified against the checkpoint config -- OK")

## 7 — The ladder

### 7a — Evaluate every variant on the frozen masked set

Each forward pass yields **two** estimators of the same masked-LM loss, at no extra GPU cost, because
both are reductions over logits the pass already computed:

- **`token_weighted`** — `reduction="sum"` over masked positions divided by the true masked-token
  count. This is the ladder's primary readout: batches carry different numbers of masked tokens, and
  it weights each token once regardless of which batch it landed in.
- **`trainer_style`** — the mean of per-batch token-mean losses, sample-weighted by batch size. This
  reproduces HF `Trainer`'s eval reduction (`losses.repeat(batch_size)` gathered, then `.mean()`),
  which is what produced colab_11's validation curve.

Computing both is what makes rung 0 a **real reproduction check** rather than a rubber stamp. Against
`trainer_style`, three of the four things that could separate this eval from colab_11's are now
matched rather than merely acknowledged:

1. **Estimator** — matched, by construction above.
2. **Batch size** — matched: `EVAL_BATCH = 8` is HF's default and colab_11 never overrode it. This
   matters beyond arithmetic, since batch geometry determines the padded width the collator draws its
   bernoulli mask over.
3. **Precision** — already matched, and the earlier claim that it was not is withdrawn. colab_11 ran
   `bf16=torch.cuda.is_bf16_supported()` (`True` on an A100) and `Trainer` wraps its eval forward in
   that autocast, so its logits were bf16 and its cross-entropy was computed in fp32 — because under
   CUDA autocast `nll_loss` is in `AT_FORALL_FP32` and `log_softmax` in `AT_FORALL_FP32_SET_OPT_DTYPE`,
   so the loss is promoted regardless of the logits' dtype. This cell does the same thing explicitly:
   bf16 autocast on the forward, fp32 cross-entropy on the gathered masked rows.
4. **Mask draw** — **the one genuine residual.** colab_11's eval masks are not recoverable (it drew
   them inside two dataloader workers on training-RNG state this notebook cannot reconstruct). This
   is precisely the quantity §7b measures, so rung 0's delta is interpretable against §7b's floor
   rather than against a made-up band.

A note on what does *not* cancel: bf16 rounding error is **not** common-mode across the ladder. The
precision *setting* is shared, but each variant carries different weights and so its own error
realisation. It is tolerable here on magnitude, not on symmetry — per-token error of order 2e-2 with
random sign, averaged over roughly 7M masked tokens, lands near 1e-5, well below the 5e-3 effect.
That is the actual argument; "identical precision, therefore cancels" was not.

The arc's real gate remains §7b: the v1→v2 gap is ca. 0.0049, and if the mask-draw spread is of that
order then colab_11's curve was never readable and there is no decoupling to explain. Attribution
(rungs A/B/C as a fraction of that gap) is therefore deferred to §7c, after the floor is known.


In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def eval_masked_loss(model, batches):
    """Both estimators from a single forward pass over `batches`.

    CE is gathered over masked rows only (ca. 15% of tokens). Scoring the full [B, T, V] logit tensor
    in fp32 would allocate several GB at T=4096 plus as much again inside log_softmax; gathering first
    keeps the fp32 upcast to the rows that actually carry a label.

    token_weighted : sum(CE) / n_masked          -- primary; weights every masked token once
    trainer_style  : sum(loss_b * rows_b) / sum(rows_b) where loss_b is batch b's token-mean CE
                     -- reproduces HF Trainer's eval_loss reduction, comparable to colab_11
    """
    model.eval().to(DEVICE)
    sum_ce = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=-100)
    tot_loss, tot_tok = 0.0, 0
    wsum, wrows = 0.0, 0
    for b in tqdm(batches, leave=False):
        ids  = b["input_ids"].to(DEVICE)
        att  = b["attention_mask"].to(DEVICE)
        lab  = b["labels"].to(DEVICE).view(-1)
        keep = lab != -100
        n_b  = int(keep.sum())
        assert n_b > 0, "a batch carries no masked tokens -- trainer_style's token-mean is undefined"
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            logits = model(input_ids=ids, attention_mask=att).logits
        sel = logits.view(-1, logits.size(-1))[keep].float()
        s = float(sum_ce(sel, lab[keep]))
        tot_loss += s; tot_tok += n_b
        rows_b = int(ids.shape[0])
        wsum  += (s / n_b) * rows_b; wrows += rows_b
        del logits, sel
    return {"token_weighted": tot_loss / tot_tok, "trainer_style": wsum / wrows,
            "n_masked_tokens": tot_tok, "n_batches": len(batches)}

LADDER = {}
for name, adapter, pats in VARIANTS:
    model, zeroed, _ = load_variant(adapter, pats)
    r = eval_masked_loss(model, frozen_batches)
    r["lora_params_zeroed"] = sum(zeroed.values())
    LADDER[name] = r
    print(f"{name:22s} token_weighted {r['token_weighted']:.6f} | "
          f"trainer_style {r['trainer_style']:.6f}  ({r['n_masked_tokens']:,} masked tokens)")
    # checkpoint after every variant: this loop is the expensive part of the run, and a disconnect
    # at the last variant must not throw away the ones already paid for.
    with open(RESULTS_PATH + ".partial", "w") as f:
        json.dump({"ladder": LADDER, "note": "partial -- §7a in progress"}, f, indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()

ntoks = {v["n_masked_tokens"] for v in LADDER.values()}
assert len(ntoks) == 1, f"variants saw different masked-token counts {ntoks} -- the set is not frozen"

print("\n--- rung 0a: does this eval reproduce colab_11's curve? (trainer_style estimator) ---")
REPRO = {}
for arm in ("v1", "v2"):
    got, ref = LADDER[f"{arm}_intact"]["trainer_style"], REF_VAL_FINAL[arm]
    REPRO[arm] = {"recomputed_trainer_style": got, "colab_11_step2000": ref, "delta": got - ref}
    print(f"  {arm}: recomputed {got:.6f}  vs colab_11 step-2000 {ref:.6f}  (delta {got-ref:+.6f})")
REPRO["residual_difference"] = "mask draw only (estimator, batch size and precision are matched)"
REPRO["interpretable_against"] = "§7b mask-draw spread"
print("  Estimator, batch size and precision are matched to colab_11; the mask draw is the only")
print("  residual, so §7b's spread -- not a fixed band -- is what these deltas must be read against.")

print("\n--- rung 0b: is there a gain to attribute? ---")
d_cpt  = LADDER["base"]["token_weighted"] - LADDER["v2_intact"]["token_weighted"]
d_v2v1 = LADDER["v1_intact"]["token_weighted"] - LADDER["v2_intact"]["token_weighted"]
REF_GAP = REF_VAL_FINAL["v1"] - REF_VAL_FINAL["v2"]
print(f"base - v2      (what CPT bought at all)   : {d_cpt:+.6f}")
print(f"v1   - v2      (the gap under attribution): {d_v2v1:+.6f}   [token_weighted]")
print(f"  same gap, trainer_style estimator       : "
      f"{LADDER['v1_intact']['trainer_style'] - LADDER['v2_intact']['trainer_style']:+.6f}")
print(f"  colab_11 step-2000 val gap              : {REF_GAP:+.6f}")
print(f"  colab_11 train_loss gap (context)       : {REF_TRAIN_LOSS['v1'] - REF_TRAIN_LOSS['v2']:+.6f}")

print("\n--- rungs A/B/C: absolute ablation cost (attribution deferred to §7c) ---")
for rung, name in [("A", "v2_minus_head"), ("B", "v2_minus_last_layer"), ("C", "v2_minus_both")]:
    cost = LADDER[name]["token_weighted"] - LADDER["v2_intact"]["token_weighted"]
    LADDER[name]["ablation_cost"] = cost
    print(f"  {rung}  {name:22s} cost {cost:+.6f}")
print("  Costs as a FRACTION of the v1->v2 gap are computed in §7c, once §7b has established")
print("  whether that gap is larger than the noise floor. Dividing by it here would print a")
print("  precise-looking percentage of a quantity that may be indistinguishable from zero.")


### 7b — Measurement noise floor: v2 across independent mask draws

`v2_intact` re-evaluated under fresh mask draws, weights held fixed. The spread is what this
instrument cannot resolve. Any ablation cost in §7a smaller than this spread is unreadable.

This bounds **measurement** noise only. It does **not** bound training-seed variance — v1 and v2 are
single runs on diverging RNG trajectories, and separating those would need retraining. So a v1→v2 gap
that clears this floor is *measured* reliably; it is still not thereby shown to be reproducible.

In [ ]:
NOISE = {}
if N_MASK_SEEDS > 1:
    model, _, _ = load_variant(V2_ADAPTER, ())
    for ms in MASK_SEEDS:
        batches = frozen_batches if ms == MASK_SEEDS[0] else build_masked_batches(val_ds, EVAL_BATCH, ms)
        r = eval_masked_loss(model, batches)
        NOISE[ms] = r
        print(f"mask seed {ms}: token_weighted {r['token_weighted']:.6f} | "
              f"trainer_style {r['trainer_style']:.6f}")
        if ms != MASK_SEEDS[0]:
            del batches; gc.collect()
        with open(RESULTS_PATH + ".partial", "w") as f:
            json.dump({"ladder": LADDER, "noise": {str(k): v for k, v in NOISE.items()},
                       "note": "partial -- §7b in progress"}, f, indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()

    # Free end-to-end determinism check: the first seed re-evaluates v2_intact on the very batches
    # §7a already scored, but from a model reloaded and re-merged from scratch. Agreement proves the
    # whole load -> attach -> merge -> eval path is reproducible across a fresh cycle. Tolerance is
    # 1e-6, not exact equality: cuBLAS GEMM algorithm selection can depend on workspace/free-memory
    # state, which genuinely differs between §7a (five prior model loads) and here, and a spurious
    # abort would cost the whole run. 1e-6 is still ca. 1000x below the effect under study.
    _r0 = NOISE[MASK_SEEDS[0]]["token_weighted"]
    _r1 = LADDER["v2_intact"]["token_weighted"]
    assert abs(_r0 - _r1) < 1e-6, (
        f"v2_intact on the frozen batches gave {_r0:.8f} here vs {_r1:.8f} in §7a -- the "
        f"load/merge/eval path is not reproducible, so no rung of the ladder is trustworthy.")
    print("reload determinism verified: v2_intact reproduces §7a to 1e-6 -- OK")

    NOISE_SUMMARY = {"seeds": MASK_SEEDS, "n_draws": len(NOISE)}
    for est in ("token_weighted", "trainer_style"):
        vals = np.array([NOISE[ms][est] for ms in MASK_SEEDS])
        rng_, sd_ = float(vals.max() - vals.min()), float(vals.std(ddof=1))
        NOISE_SUMMARY[est] = {"losses": {str(k): NOISE[k][est] for k in NOISE},
                              "sd": sd_, "range": rng_}
        print(f"\n{est}: sd {sd_:.6f} | range {rng_:.6f}  (n={len(vals)} draws)")
    # The floor used to gate attribution is the token_weighted sd, since the ladder is measured on
    # that estimator. sd (not range) because range is a strongly downward-biased spread estimator,
    # which would make the gap look more readable than it is.
    NOISE_FLOOR = NOISE_SUMMARY["token_weighted"]["sd"]
    print(f"\nnoise floor for §7c gating = token_weighted sd = {NOISE_FLOOR:.6f}")
    print(f"CAUTION: n={len(NOISE)} draws. sd from n=4 carries ca. 40% relative error and the range")
    print("  is downward-biased, so treat any 'Nx the floor' ratio as order-of-magnitude only.")
    print(f"v1->v2 gap {d_v2v1:+.6f} vs floor {NOISE_FLOOR:.6f} -> "
          f"{abs(d_v2v1)/NOISE_FLOOR:.1f}x (order-of-magnitude)" if NOISE_FLOOR > 0 else "")
else:
    print("*** N_MASK_SEEDS=1: noise floor NOT measured -- the arc's gate is absent ***")
    NOISE_SUMMARY = {"skipped": f"N_MASK_SEEDS={N_MASK_SEEDS}"}
    NOISE_FLOOR = None


### 7c — Verdicts, gated on the measured floor

Both of this notebook's verdicts need §7b's noise floor as their denominator, so both are made here
rather than beside the numbers that produce them.

**Rung 0a's verdict.** §7a printed how far the recomputed `trainer_style` losses sit from colab_11's
step-2000 values. Estimator, batch size and precision are matched, so the mask draw is the only thing
left between them — and both sides are a single draw from that distribution, evaluated on identical
weights. Their difference therefore has roughly √2 times the per-draw spread §7b measures, which
gives a principled band instead of an invented one: a delta of a few √2·sd is ordinary, and a delta
far outside it means something structural differs (wrong val set, wrong masking rate, broken token
handling) and nothing below is trustworthy.

**Attribution.** Rungs A/B/C as a fraction of the v1→v2 gap, computed only if that gap clears the
floor. Dividing by an unreadable denominator would print a confident percentage of a quantity
indistinguishable from zero — at a gap of +2e-5 it would report a rung as "4000% of the gap" and
write that into the permanent audit record. If the gate closes, that is a **result, not a failure**:
it is the arc's live hypothesis, and it would mean colab_11's curve was never readable and the
loss/drift decoupling dissolves before head absorption is even a question worth asking.


In [ ]:
import math

SANITY_BREACHED, ATTRIBUTION_GATED = False, None

print("--- rung 0a verdict: reproduction of colab_11's curve ---")
if NOISE_FLOOR is None:
    SANITY_BREACHED = False
    REPRO["verdict"] = "not assessable -- noise floor not measured"
    print("  not assessable: §7b did not run, so there is no floor to read the deltas against.")
else:
    # Both sides are one mask draw on identical weights, so their difference carries ca. sqrt(2)x
    # the per-draw spread. 3 sigma on that is the band; it is derived from this run's own measured
    # noise rather than chosen to fit the deltas it judges.
    sd_ts = NOISE_SUMMARY["trainer_style"]["sd"]
    band  = 3 * math.sqrt(2) * sd_ts
    REPRO["band_3sigma_sqrt2"] = band
    for arm in ("v1", "v2"):
        d = abs(REPRO[arm]["delta"])
        hit = d > band
        SANITY_BREACHED |= hit
        REPRO[arm]["within_band"] = not hit
        print(f"  {arm}: |delta| {d:.6f} vs band {band:.6f} "
              f"({'OUTSIDE -- structural' if hit else 'within -- reproduces'})")
    REPRO["verdict"] = "structural difference" if SANITY_BREACHED else "reproduces colab_11"
    if SANITY_BREACHED:
        print("  *** REPRODUCTION FAILED: this eval differs STRUCTURALLY from colab_11's. Suspect a")
        print("      different val set, masking rate, or token handling. Every rung below inherits")
        print("      the problem, and RESULTS['status'] records this run as caveated. ***")

print("\n--- attribution ---")
if NOISE_FLOOR is None:
    ATTRIBUTION_GATED = "noise floor not measured (N_MASK_SEEDS=1)"
elif abs(d_v2v1) <= NOISE_FLOOR:
    ATTRIBUTION_GATED = (f"v1->v2 gap {d_v2v1:+.6f} does not clear the noise floor {NOISE_FLOOR:.6f} "
                         f"-- there is no readable gain to attribute")

if ATTRIBUTION_GATED:
    print("*** ATTRIBUTION SKIPPED ***")
    print("   ", ATTRIBUTION_GATED)
    print("    Rungs A/B/C keep their absolute costs from §7a; frac_of_v2_v1_gap is null.")
    print("    Read this as a RESULT, not an error: if the gap is unreadable, colab_11's val curve")
    print("    was never readable either, and the loss/drift decoupling this arc exists to explain")
    print("    dissolves -- head absorption does not arise as a question.")
    for name in ("v2_minus_head", "v2_minus_last_layer", "v2_minus_both"):
        LADDER[name]["frac_of_v2_v1_gap"] = None
else:
    print(f"v1->v2 gap {d_v2v1:+.6f} clears the floor {NOISE_FLOOR:.6f} "
          f"({abs(d_v2v1)/NOISE_FLOOR:.1f}x -- order-of-magnitude only at n={NOISE_SUMMARY['n_draws']})")
    print("\nrung   variant                 cost        frac of gap   readable?")
    for rung, name in [("A", "v2_minus_head"), ("B", "v2_minus_last_layer"), ("C", "v2_minus_both")]:
        cost = LADDER[name]["ablation_cost"]
        frac = cost / d_v2v1
        LADDER[name]["frac_of_v2_v1_gap"] = frac
        LADDER[name]["cost_clears_floor"] = bool(abs(cost) > NOISE_FLOOR)
        readable = "yes" if abs(cost) > NOISE_FLOOR else "NO -- below floor, treat as 0"
        print(f"  {rung}    {name:22s} {cost:+.6f}   {frac:8.1%}     {readable}")

    print("\nReading: rung C (head + layer 11) is detector #1's blind spot -- 9.2% of v2's added")
    print("capacity. A small C means v2's gain landed where detector #1 could see it, and the")
    print("loss/drift decoupling needs an explanation other than head absorption.")
    print("Rungs are UPPER BOUNDS and need not sum to C: the encoder trained with every delta")
    print("present, so removing one at a time cannot decompose a jointly-trained fit.")


## 8 — Save + handoff

### 8a — Write results, append the audit trace, print commit commands

A smoke run writes only to `_SMOKE`-suffixed paths and does **not** touch `audit_report.json`, so it
can never contaminate the audit trail.

In [ ]:
import shlex

# status is DERIVED, never asserted. audit_report.json is re-read by every later notebook, so a
# run whose reproduction check breached, or whose gate never ran, must say so in the artifact itself
# -- not only in a print buried in a long Colab log.
_flags = []
if SMOKE:                        _flags.append("smoke")
if NOISE_SUMMARY.get("skipped"): _flags.append("noise_floor_not_measured")
if SANITY_BREACHED:              _flags.append("reproduction_failed")
if ATTRIBUTION_GATED:            _flags.append("attribution_gated")
STATUS = "computed" if not _flags else "computed_with_caveats:" + ",".join(_flags)
print("status:", STATUS)

RESULTS = {
    "status": STATUS, "caveats": _flags, "date": TODAY, "smoke": SMOKE, "run_tag": RUN_TAG,
    "question": "where did colab_11 v2's loss gain land, and was detector #1 positioned to see it?",
    "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
    "surface": "val", "n_val_cells": int(val.n_obs), "eval_batch": EVAL_BATCH, "mlm_prob": MLM_PROB,
    "peft_version": peft.__version__, "transformers_version": transformers.__version__,
    "loss_estimators": {
        "token_weighted": "sum(CE over masked positions)/n_masked -- the ladder's primary readout",
        "trainer_style": "sample-weighted mean of per-batch token-mean CE -- reproduces HF "
                         "Trainer's eval reduction, the estimator colab_11's curve used",
    },
    "precision": "bf16 autocast forward + fp32 cross-entropy -- matches colab_11's Trainer",
    "attribution_gated": ATTRIBUTION_GATED, "noise_floor_token_weighted": NOISE_FLOOR,
    "emb_layer_convention": "emb_layer=-1 extracts layer L-2 output; layer L-1 + MLM head are downstream",
    "geometry": {"n_layers": L, "hidden": H, "intermediate": I, "lora_r": R,
                 "lora_params_head": EXP_HEAD, "lora_params_last_layer": EXP_LAST,
                 "lora_params_blind_spot": EXP_BOTH, "v2_trainable": V2_TRAINABLE,
                 "blind_spot_frac_of_v2": EXP_BOTH / V2_TRAINABLE},
    "ablation_audit": ABLATION_AUDIT,
    "ladder": LADDER,
    "noise_summary": NOISE_SUMMARY,
    "ref_train_loss": REF_TRAIN_LOSS,
    "ref_val_loss_curve": {str(k): v for k, v in REF_VAL_LOSS.items()},
    "reproduction_check_vs_colab_11": REPRO,
    "limits": [
        "ablation is not decomposition: the encoder trained with the head delta present, so each "
        "rung is an upper bound on its region's contribution and rungs need not sum to C",
        "does not bound training-seed variance: v1/v2 are N=1 on diverging RNG trajectories",
        "colab_11's 8-checkpoint val curve is 8 samples of the same 2 trajectories, not 8 "
        "independent draws -- 'v2 wins 8/8' carries almost no evidential weight",
        "v1/v2 consumed different training RNG, so their eval masks likely differed; part of the "
        "reference curve's 0.0031-0.0049 gap may be mask noise, which §7b bounds",
        "rung 0a reproduces colab_11 on the trainer_style estimator with estimator, batch size (8) "
        "and precision (bf16 forward + fp32 CE) matched; the mask draw is the only residual, and its "
        "spread from §7b sets the band the deltas are judged against",
        "the noise floor rests on n=4 mask draws: sd carries ca. 40% relative error at that n and "
        "range is downward-biased, so any 'Nx the floor' ratio is order-of-magnitude only",
        "bf16 rounding error is NOT common-mode across variants (different weights -> different error "
        "realisations); it is tolerable on magnitude (ca. 1e-5 after averaging over ca. 7M masked "
        "tokens), not because it cancels",
        "peft/transformers/datasets are asserted in §2a against colab_11's recorded pip freeze "
        "(peft 0.19.1, transformers 4.57.0, datasets 5.0.0), so the merge semantics that produced "
        "these numbers match the ones that produced the adapters; torch and numpy ride Colab's base "
        "image and are recorded but not pinned",
    ],
}
with open(RESULTS_PATH, "w") as f:
    json.dump(RESULTS, f, indent=2)
print("results ->", RESULTS_PATH)

if os.path.exists(RESULTS_PATH + ".partial"):
    os.remove(RESULTS_PATH + ".partial")   # full results landed; the crash breadcrumb is moot

if SMOKE:
    print("\n*** SMOKE RUN -- audit_report.json NOT touched, nothing to commit ***")
else:
    AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
    report[AUDIT_KEY] = RESULTS
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("audit trace appended ->", AUDIT_REPORT_PATH, f"[{AUDIT_KEY}]")

    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    print("  NOTE: these live in the EPHEMERAL Colab clone. Download them from the Colab file browser")
    print("  into the WSL clone first, or the git add below stages nothing.")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          "'diag: LoRA ablation ladder -- attribute colab_11 v2 fit vs detector #1 blind spot'")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

### Carried forward

- `cpt_head_ablation_results.json` on Drive + the `geneformer_cpt_head_ablation` block in
  `outputs/audit_report.json` — the ladder, the ablation audit, and the noise floor.
- Both LoRA adapters remain the only durable CPT artifacts; every variant here was re-merged from
  base, and no merged checkpoint is persisted.
- Whatever rung 0 says governs what the B2 arc does next: a v1→v2 gap at the noise floor retires the
  loss/drift decoupling as a phenomenon; a gap that clears it hands the attribution to rungs A–C, and
  rung C specifically to the question of whether detector #1's extraction point needs to move.